# Chapter 02. Machine Learning Fundamentals

입력 데이터, Linear Regression, Loss, Gradient Descent, Feature Scaling의 기본 계산을 실행합니다. Notebook의 데이터는 실습용 합성 데이터이며 CSV 업로드나 GPU가 필요하지 않습니다.

## 1. 실행 준비

Colab이나 Jupyter Notebook에서 코드 셀을 순서대로 실행합니다. 라이브러리가 설치되어 있지 않다면 설치 셀의 주석을 해제합니다.

In [ ]:
# %pip install numpy pandas matplotlib scikit-learn

## 2. Sample, Feature, Target

행은 Sample, 열은 Feature입니다. 각 Sample에 대응하는 Target은 별도 배열로 저장합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X = np.array([[2, 1], [4, 0], [6, 2]], dtype=float)
y = np.array([9, 17, 27], dtype=float)
print("X shape:", X.shape)
print("y shape:", y.shape)

## 3. Loss 계산

MAE는 절대 오차의 평균, MSE는 제곱 오차의 평균입니다.

In [ ]:
y_true = np.array([3, 5, 7], dtype=float)
y_pred = np.array([2, 7, 8], dtype=float)
print(f"MAE: {mean_absolute_error(y_true, y_pred):.3f}")
print(f"MSE: {mean_squared_error(y_true, y_pred):.3f}")

## 4. Baseline과 Linear Regression

거리와 경유지 수로부터 소요 시간을 생성합니다. 모델에는 먼저 거리만 사용하고, Baseline과 같은 Test Set에서 MAE를 비교합니다.

In [ ]:
rng = np.random.default_rng(20261108)
distance = rng.uniform(0.5, 15.0, 180)
stops = rng.integers(0, 7, 180)
duration = 4.5 + 3.2 * distance + 1.8 * stops + rng.normal(0, 2.0, 180)
X_delivery = pd.DataFrame({"distance_km": np.round(distance, 3)})
y_delivery = np.round(duration, 2)
X_train, X_test, y_train, y_test = train_test_split(
    X_delivery, y_delivery, test_size=0.25, random_state=42
)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
regressor = LinearRegression().fit(X_train, y_train)
pred = regressor.predict(X_test)
print(f"Baseline MAE: {mean_absolute_error(y_test, baseline.predict(X_test)):.3f} min")
print(f"Linear MAE: {mean_absolute_error(y_test, pred):.3f} min")
print(f"Weight: {regressor.coef_[0]:.3f} min/km")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.2))
ordered = X_test.sort_values("distance_km")
ax.scatter(X_test["distance_km"], y_test, alpha=.7, label="Test samples")
ax.plot(ordered["distance_km"], regressor.predict(ordered), label="Prediction")
ax.set(xlabel="Distance (km)", ylabel="Duration (min)", title="Linear Regression")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Gradient Descent

MSE의 기울기를 계산하고 Weight와 Bias를 반복해서 갱신합니다. 갱신 후 MSE를 기록하여 감소 과정을 확인합니다.

In [ ]:
grad_rng = np.random.default_rng(42)
x = np.linspace(-2, 2, 41)
y = 1.5 * x + 2.8 + grad_rng.normal(0, 0.25, x.size)
w, b, alpha = 0.0, 0.0, 0.05
losses = []
for _ in range(250):
    error = w * x + b - y
    grad_w = 2 * np.mean(error * x)
    grad_b = 2 * np.mean(error)
    w -= alpha * grad_w
    b -= alpha * grad_b
    losses.append(np.mean((w * x + b - y) ** 2))
print(f"Weight: {w:.4f}, Bias: {b:.4f}, MSE: {losses[-1]:.4f}")
fig, ax = plt.subplots(figsize=(6.6, 4.1))
ax.plot(np.arange(1, len(losses) + 1), losses)
ax.set(xlabel="Iteration", ylabel="MSE", title="Gradient Descent")
fig.tight_layout()
plt.show()

## 6. Feature Scaling과 KNN

진동은 0~4mm/s, 조도는 100~12,000lx의 값으로 만듭니다. 진동이 Label과 관련되도록 설정한 합성 데이터이므로, 스케일링 전후의 차이를 확인할 수 있습니다.

In [ ]:
vibration = rng.uniform(0, 4, 240)
ambient = rng.uniform(100, 12000, 240)
fault = (vibration + rng.normal(0, .22, 240) >= 2.0).astype(int)
X_sensor = pd.DataFrame({"vibration_mm_s": vibration, "ambient_lux": ambient})
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sensor, fault, test_size=.25, random_state=42, stratify=fault
)
raw_knn = KNeighborsClassifier(n_neighbors=7).fit(X_train_s, y_train_s)
scaled_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
scaled_knn.fit(X_train_s, y_train_s)
print(f"Raw Accuracy: {accuracy_score(y_test_s, raw_knn.predict(X_test_s)):.1%}")
print(f"Scaled Accuracy: {accuracy_score(y_test_s, scaled_knn.predict(X_test_s)):.1%}")

## 7. Data Leakage

예측이 끝난 뒤에만 확인되는 값은 실제 입력 Feature로 사용할 수 없습니다. 아래 비교는 Data Leakage가 평가 결과를 왜곡할 수 있음을 보여주는 합성 실험입니다.

In [ ]:
n = 320
temperature = rng.uniform(9, 33, n)
area = rng.uniform(40, 250, n)
occupancy = rng.integers(1, 13, n)
energy = 1.1 * area + 2.5 * np.abs(temperature - 21) + 8 * occupancy + rng.normal(0, 10, n)
finalized = energy + rng.normal(0, .12, n)  # 측정 종료 후 생성된 값
frame = pd.DataFrame({"temperature_c": temperature, "floor_area_m2": area,
                      "occupancy": occupancy, "meter_finalized_kwh": finalized})
valid = ["temperature_c", "floor_area_m2", "occupancy"]
leaked = valid + ["meter_finalized_kwh"]
train_idx, test_idx = train_test_split(np.arange(n), test_size=.25, random_state=42)
valid_model = LinearRegression().fit(frame.iloc[train_idx][valid], energy[train_idx])
leaked_model = LinearRegression().fit(frame.iloc[train_idx][leaked], energy[train_idx])
valid_mae = mean_absolute_error(energy[test_idx], valid_model.predict(frame.iloc[test_idx][valid]))
leaked_mae = mean_absolute_error(energy[test_idx], leaked_model.predict(frame.iloc[test_idx][leaked]))
print(f"Available features MAE: {valid_mae:.3f} kWh")
print(f"Leaked feature MAE: {leaked_mae:.3f} kWh")

## 8. 직접 확인하기

다음 실험을 수행하고 결과를 기록합니다.

1. 배송 시간 모델에 `stops`를 추가하고 MAE를 비교합니다.
2. Gradient Descent의 Learning Rate를 `0.01`과 `0.2`로 바꾸고 Loss 곡선을 비교합니다.
3. 센서 데이터에서 `ambient_lux`를 제거하고 KNN의 Accuracy를 확인합니다.
4. 예측 종료 후 생성되는 계량 값을 제외해야 하는 이유를 설명합니다.

관련 수식과 프로젝트별 요구사항은 Chapter 02의 `README.md`에 있습니다.